In [1]:
import json
import pandas as pd

In [2]:
df_left_right = pd.read_csv('3DSR_left_right_eval.csv')
df_left_right_filtered = df_left_right[df_left_right['keep'] == 1]
indices_to_analyze = df_left_right_filtered['index'].tolist()
indices_to_analyze[:3]

['C718L2MO', 'C718L2MO-flip', 'N7DZ6RW8']

In [3]:
with open('left_right_answers.json', 'r') as file:
    answers = json.load(file)

print(answers[:10])

[{'question_id': 'Q793LOEX', 'answer': 'left'}, {'question_id': 'Q793LOEX-flip', 'answer': 'right'}, {'question_id': 'IFE0O19V', 'answer': 'left'}, {'question_id': 'IFE0O19V-flip', 'answer': 'right'}, {'question_id': '3XH5DWRM', 'answer': 'left'}, {'question_id': '3XH5DWRM-flip', 'answer': 'right'}, {'question_id': 'ZWS322ON', 'answer': 'right'}, {'question_id': 'ZWS322ON-flip', 'answer': 'left'}, {'question_id': 'HGL6CU3T', 'answer': 'left'}, {'question_id': 'HGL6CU3T-flip', 'answer': 'right'}]


In [4]:
# take only items where question_id does not contain "flip"
left_right_answers = [ans for ans in answers if (ans['question_id'] in indices_to_analyze and "flip" not in ans['question_id'])]

In [5]:
left_right_answers = [ans['answer'] for ans in answers]
print(left_right_answers[:10])

['left', 'right', 'left', 'right', 'left', 'right', 'right', 'left', 'left', 'right']


### Base results

In [6]:
with open('answers/answers_base.jsonl', 'r') as file:
    base_answers = [json.loads(line) for line in file]
print(base_answers[:10])

[{'question_id': 'Q793LOEX', 'prompt': "A woman in purple cloth and a kitchen paper towel are shown in the image. Which side of the woman in purple cloth is the kitchen paper towel on from the woman in purple cloth's perspective?", 'text': "From the woman in purple cloth's perspective, the kitchen paper towel is on her right side.", 'answer_id': 'mMRVkBdt4djBh5VcgHUEE3', 'model_id': 'llava-v1.5-13b', 'metadata': {}}, {'question_id': 'Q793LOEX-flip', 'prompt': "A woman in purple cloth and a kitchen paper towel are shown in the image. Which side of the woman in purple cloth is the kitchen paper towel on from the woman in purple cloth's perspective?", 'text': "From the woman in purple cloth's perspective, the kitchen paper towel is on her right side.", 'answer_id': 'mAfxmvznmag4iACdJ7GN5q', 'model_id': 'llava-v1.5-13b', 'metadata': {}}, {'question_id': 'IFE0O19V', 'prompt': "A bus and a blue board are shown in the image. Which side of the bus is the blue board on from the bus's perspectiv

In [7]:
base_responses = [ans['text'] for ans in base_answers if (ans['question_id'] in indices_to_analyze and "flip" not in ans['question_id'])]
print(base_responses[:10])

["From the person's perspective, the white menu board is on the right side.", "From the dog's perspective, the vase is on the left side.", "From the cat's perspective, the laptop is on the left side.", 'The bus stop board is on the left side of the man in white shirt from his perspective.', "From the armchair's perspective, the tub is on the left side.", "The bottle is on the right side of the couch from the couch's perspective.", "From the person's perspective, the bottled water is on their left side.", "From the squirrel's perspective, the trash can is on the right side.", "From the cat's perspective, the book is on the right side.", "From the dog's perspective, the spare tire is on the left side."]


In [8]:
base_answers_clean = []

for ans in base_responses:
    if 'left' in ans.lower():
        base_answers_clean.append('left')
    elif 'right' in ans.lower():
        base_answers_clean.append('right')
    else:
        base_answers_clean.append('unknown')
print(base_answers_clean[:10])

['right', 'left', 'left', 'left', 'left', 'right', 'left', 'right', 'right', 'left']


In [9]:
base_correct = [base == lr for base, lr in zip(base_answers_clean, left_right_answers)]
print(sum(base_correct) / len(base_correct))

0.4


### Text results

In [10]:
with open('answers/answers_vit_text.jsonl', 'r') as file:
    text_answers = [json.loads(line) for line in file]
print(text_answers[:10])

[{'question_id': 'Q793LOEX', 'prompt': "A woman in purple cloth and a kitchen paper towel are shown in the image. Which side of the woman in purple cloth is the kitchen paper towel on from the woman in purple cloth's perspective?", 'text': 'left', 'answer_id': 'FWnRqibajp8BAHzKnnQHVt', 'model_id': 'train_text_annealing-llava-v1.5-13b-task-lora-ViT', 'metadata': {}}, {'question_id': 'Q793LOEX-flip', 'prompt': "A woman in purple cloth and a kitchen paper towel are shown in the image. Which side of the woman in purple cloth is the kitchen paper towel on from the woman in purple cloth's perspective?", 'text': 'left', 'answer_id': 'M7AhqYNLYW9qvZHQxNGtEu', 'model_id': 'train_text_annealing-llava-v1.5-13b-task-lora-ViT', 'metadata': {}}, {'question_id': 'IFE0O19V', 'prompt': "A bus and a blue board are shown in the image. Which side of the bus is the blue board on from the bus's perspective?", 'text': 'left', 'answer_id': 'ZadxLkvLSox3ZUPdMwtAgU', 'model_id': 'train_text_annealing-llava-v1.5

In [11]:
text_responses = [ans['text'] for ans in text_answers if (ans['question_id'] in indices_to_analyze and "flip" not in ans['question_id'])]
print(text_responses[:10])

['left', 'right', 'right', 'right', 'right', 'right', 'right', 'right', 'left', 'right']


In [12]:
text_correct = [text == lr for text, lr in zip(text_responses, left_right_answers)]
print(sum(text_correct) / len(text_correct))

0.5777777777777777


#### CoT

In [13]:
with open('answers/answers_vit_text_CoT.jsonl', 'r') as file:
    text_cot_answers = [json.loads(line) for line in file]
text_cot_answers = [cot['text'] for cot in text_cot_answers if (cot['question_id'] in indices_to_analyze and "flip" not in cot['question_id'])]
print(text_cot_answers[0])

The white menu board is right of the person from the viewer's frame. The person's pose is 183.0 159.0 0.9 97.0 154.0 0.9 170.0 304.0 0.8 110.0 307.0 0.7 3 3. Since 3 lies in the 135-180 bin, the person is unaligned. After performing a mental rotation of 135-180 degrees, the white menu board is now to the left. Therefore, the white menu board is to the left from the perspective of the person.


In [14]:
text_cot_responses = [cot.split("Therefore, the ")[1].split("is to the ")[1].split(" from the")[0].strip().lower() for cot in text_cot_answers]
print(text_cot_responses[:10])

['left', 'right', 'right', 'right', 'left', 'left', 'right', 'left', 'left', 'right']


In [15]:
text_cot_correct = [text_cot == lr for text_cot, lr in zip(text_cot_responses, left_right_answers)]
print(sum(text_cot_correct) / len(text_cot_correct))

0.6666666666666666


### COCO results

In [16]:
with open('answers/answers_coco.jsonl', 'r') as file:
    coco_answers = [json.loads(line) for line in file]
print(coco_answers[:10])

[{'question_id': 'Q793LOEX', 'prompt': "A woman in purple cloth and a kitchen paper towel are shown in the image. Which side of the woman in purple cloth is the kitchen paper towel on from the woman in purple cloth's perspective?", 'text': 'right', 'answer_id': 'WHtLa3BDbopMVMAG5ZRdZm', 'model_id': 'train_perspective_annealing-llava-v1.5-13b-task-lora', 'metadata': {}}, {'question_id': 'Q793LOEX-flip', 'prompt': "A woman in purple cloth and a kitchen paper towel are shown in the image. Which side of the woman in purple cloth is the kitchen paper towel on from the woman in purple cloth's perspective?", 'text': 'right', 'answer_id': 'Pp96F8cZwy2662agpZSRB5', 'model_id': 'train_perspective_annealing-llava-v1.5-13b-task-lora', 'metadata': {}}, {'question_id': 'IFE0O19V', 'prompt': "A bus and a blue board are shown in the image. Which side of the bus is the blue board on from the bus's perspective?", 'text': 'right', 'answer_id': 'joByqZ8JYVdavQgmgXpWvv', 'model_id': 'train_perspective_anne

In [17]:
coco_responses = [ans['text'] for ans in coco_answers if (ans['question_id'] in indices_to_analyze and "flip" not in ans['question_id'])]
print(coco_responses[:10])

['right', 'right', 'left', 'right', 'left', 'right', 'right', 'right', 'left', 'left']


In [18]:
coco_correct = [coco == lr for coco, lr in zip(coco_responses, left_right_answers)]
print(sum(coco_correct) / len(coco_correct))

0.4888888888888889


#### CoT

In [22]:
with open('answers/answers_coco_CoT.jsonl', 'r') as file:
    coco_cot_answers = [json.loads(line) for line in file]
coco_cot_answers = [cot['text'] for cot in coco_cot_answers if (cot['question_id'] in indices_to_analyze and "flip" not in cot['question_id'])]
print(coco_cot_answers[0])

The white menu board is right of the person from the viewer's frame. The person's pose is  <POSE_START> <SHOULDER_L> <X_175> <Y_159> <SHOULDER_R> <X_98> <Y_158> <HIP_L> <X_173> <Y_301> <HIP_R> <X_110> <Y_299> <ORIENT_START> <YAW_3> <TORSO_W_3> <ORIENT_END> <POSE_END> . Since  <YAW_3> lies in the 135-180 bin, the person is unaligned. After performing a mental rotation of 135-180 degrees, the white menu board is now to the left. Therefore, the white menu board is to the left from the perspective of the person.


In [23]:
coco_cot_responses = [cot.split("Therefore, the ")[1].split("is to the ")[1].split(" from the")[0].strip().lower() for cot in coco_cot_answers]
print(coco_cot_responses[:10])

['left', 'right', 'right', 'left', 'right', 'right', 'right', 'left', 'left', 'left']


In [24]:
coco_cot_correct = [coco_cot == lr for coco_cot, lr in zip(coco_cot_responses, left_right_answers)]
print(sum(coco_cot_correct) / len(coco_cot_correct))

0.5777777777777777


### ViT results

In [19]:
with open('answers/answers_vit.jsonl', 'r') as file:
    vit_answers = [json.loads(line) for line in file]
print(vit_answers[:10])

[{'question_id': 'Q793LOEX', 'prompt': "A woman in purple cloth and a kitchen paper towel are shown in the image. Which side of the woman in purple cloth is the kitchen paper towel on from the woman in purple cloth's perspective?", 'text': 'left', 'answer_id': 'dFFrzmduKtGRL36UHdKPHd', 'model_id': 'train_perspective_annealing-llava-v1.5-13b-task-lora-ViT', 'metadata': {}}, {'question_id': 'Q793LOEX-flip', 'prompt': "A woman in purple cloth and a kitchen paper towel are shown in the image. Which side of the woman in purple cloth is the kitchen paper towel on from the woman in purple cloth's perspective?", 'text': 'left', 'answer_id': 'BUNS49zEGnABrkrziQm9Dy', 'model_id': 'train_perspective_annealing-llava-v1.5-13b-task-lora-ViT', 'metadata': {}}, {'question_id': 'IFE0O19V', 'prompt': "A bus and a blue board are shown in the image. Which side of the bus is the blue board on from the bus's perspective?", 'text': 'left', 'answer_id': 'Kh6gews8sxmHWP8pcWno8V', 'model_id': 'train_perspective

In [20]:
vit_responses = [ans['text'] for ans in vit_answers if (ans['question_id'] in indices_to_analyze and "flip" not in ans['question_id'])]
print(vit_responses[:10])

['left', 'left', 'right', 'left', 'left', 'left', 'right', 'right', 'left', 'left']


In [21]:
vit_correct = [vit == lr for vit, lr in zip(vit_responses, left_right_answers)]
print(sum(vit_correct) / len(vit_correct))

0.4888888888888889


#### CoT

In [25]:
with open('answers/answers_vit_CoT.jsonl', 'r') as file:
    vit_cot_answers = [json.loads(line) for line in file]
vit_cot_answers = [cot['text'] for cot in vit_cot_answers if (cot['question_id'] in indices_to_analyze and "flip" not in cot['question_id'])]
print(vit_cot_answers[0])

The white menu board is right of the person from the viewer's frame. The person's pose is  <POSE_START> <SHOULDER_L> <X_178> <Y_167> <TORSO_W_1> <SHOULDER_R> <X_108> <Y_154> <TORSO_W_0> <HIP_L> <X_180> <Y_312> <TORSO_W_0> <HIP_R> <X_117> <Y_316> <YAW_7> <ORIENT_START> <CONF_2> <CONF_9> <ORIENT_END> <POSE_END> . Since YAW_2 lies in the 90-135 bin, the person is unaligned. After performing a mental rotation of 90-135 degrees, the white menu board is now to the left. Therefore, the white menu board is to the left from the perspective of the person.


In [26]:
vit_cot_responses = [cot.split("Therefore, the ")[1].split("is to the ")[1].split(" from the")[0].strip().lower() for cot in vit_cot_answers]
print(vit_cot_responses[:10])

['left', 'right', 'right', 'left', 'left', 'right', 'left', 'left', 'right', 'right']


In [27]:
vit_cot_correct = [vit_cot == lr for vit_cot, lr in zip(vit_cot_responses, left_right_answers)]
print(sum(vit_cot_correct) / len(vit_cot_correct))

0.6444444444444445
